<a href="https://colab.research.google.com/github/Amitabh-Phule/GenAi/blob/main/StudyGenie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧞‍♂️ StudyGenie – AI Powered PDF Study Assistant
---
## 📌 Concept
StudyGenie is a smart learning assistant that behaves like a “genie for studying”.
It provides **3 intelligent wishes** to help students learn faster from PDF documents.

---

## ✨ The 3 Wishes System
1. ❓ Ask Questions (AI + Retrieval based Q&A)
2. 📄 Generate Summary (Important content extraction)
3. 📘 Create Study Plan (Revision schedule generation)


### STEP 1: Setup Environment
Install required libraries for Deep Learning embeddings, FAISS search, PDF processing, and Groq LLM.

In [ ]:
!pip install -q groq PyPDF2 sentence-transformers faiss-cpu

print("✅ Environment Ready")

### STEP 2: Import Modules
Import all required libraries for RAG pipeline, embeddings, and LLM integration.

In [ ]:
from groq import Groq
from google.colab import userdata
import PyPDF2
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("✅ Imports Loaded")

### STEP 3: API Configuration
Securely load Groq API key and initialize LLM model.

In [ ]:
API_KEY = userdata.get("StudyGenie")

if not API_KEY:
    raise Exception("Missing API Key")

client = Groq(api_key=API_KEY)

MODEL_NAME = "llama-3.1-8b-instant"

print("✅ Groq Connected")

### STEP 4: Deep Learning Model
Load Sentence Transformer model for generating semantic embeddings.

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding Model Loaded")

### STEP 5: PDF Input
Upload PDF file and extract raw text.

In [ ]:
from google.colab import files
import PyPDF2

pdf_text = ""

def load_pdf():
    global pdf_text

    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

    text = ""
    total_pages = 0
    extracted_pages = 0

    with open(file_name, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        total_pages = len(reader.pages)

        print(f"📄 Total Pages: {total_pages}")

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text and page_text.strip():
                text += page_text + "\n"
                extracted_pages += 1

    pdf_text = text.strip()

    print(f"📊 Extracted Pages: {extracted_pages}/{total_pages}")
    print(f"📏 Total Text Length: {len(pdf_text)}")

    if len(pdf_text) < 50:
        print("⚠️ Warning: PDF may be scanned or image-based")
    else:
        print("✅ PDF Loaded Successfully")

### STEP 6: Text Chunking
Split PDF text into smaller chunks for better retrieval.

In [ ]:
chunks = []

def create_chunks(text, size=250):
    words = text.split()

    return [
        " ".join(words[i:i+size])
        for i in range(0, len(words), size)
    ]

def build_chunks():
    global chunks

    if not pdf_text or len(pdf_text.strip()) == 0:
        raise Exception("❌ PDF is empty. Upload valid file first.")

    chunks = create_chunks(pdf_text)

    chunks = chunks[:50]  # limit cost + memory
print(f"✅ STEP 6 DONE: Chunks created = {len(chunks)}")

### STEP 7: FAISS Index
Convert text chunks into embeddings and store in FAISS vector database.

In [ ]:
def build_faiss():
    global chunk_embeddings, index

    if len(chunks) == 0:
        raise Exception("❌ No chunks found. Run build_chunks first.")

    # Step 1: embeddings
    chunk_embeddings = embed_model.encode(chunks)

    # Step 2: convert to numpy
    chunk_embeddings = np.array(chunk_embeddings)

    # Step 3: force 2D shape (VERY IMPORTANT)
    if len(chunk_embeddings.shape) == 1:
        chunk_embeddings = chunk_embeddings.reshape(1, -1)

    chunk_embeddings = chunk_embeddings.astype("float32")

    # Step 4: create FAISS index dynamically
    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)

    # Step 5: add vectors
    index.add(chunk_embeddings)

    # Step 6: SAFE PRINT (INSIDE FUNCTION)
    print("✅ FAISS Ready")
    print("📊 Shape:", chunk_embeddings.shape)

### STEP 8: Semantic Retrieval
Fetch most relevant chunks using vector similarity search.

In [ ]:
def retrieve(query, k=3):
    q_vec = embed_model.encode([query]).astype("float32")

    _, indices = index.search(q_vec, k)

    return [chunks[i] for i in indices[0]]

### STEP 9: Generative AI Layer
Use Groq LLM to generate final responses.

In [ ]:
def groq_call(prompt, max_tokens=200):
    try:
        res = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens
        )
        return res.choices[0].message.content

    except Exception as e:
        return f"Error: {e}"

print("✅ LLM Ready")

### STEP 10: Question Answering
Answer user queries using retrieved context + LLM reasoning.

In [ ]:
def ask_question():
    q = input("Ask: ")

    context = retrieve(q)

    prompt = f"""
Answer using context only.

Context:
{context}

Question:
{q}
"""

    print("\n💡 Answer:\n")
    print(groq_call(prompt, 180))

### STEP 11: Summary Generation
Generate concise summary from PDF content.

In [ ]:
def summary():
    context = chunks[:6]

    prompt = f"Summarize clearly:\n{context}"

    print("\n📄 Summary:\n")
    print(groq_call(prompt, 200))

### STEP 12: Study Plan
Generate a simple, structured study plan based on the PDF content.

In [ ]:
def study_plan():
    context = chunks[:8]

    prompt = f"""
Create a simple study plan from the content below.

Rules:
- Keep it short and practical
- Use headings like: Concepts, Revision Points, Important Topics
- Do NOT use fixed days like Day 1, Day 2
- Make it easy for exam preparation

Content:
{context}
"""

    print("\n📘 Study Plan:\n")
    print(groq_call(prompt, 220))

print("\n✅ Study Plan Generated")

### STEP 13: Run System
Main interface for interacting with StudyGenie.

In [ ]:
from IPython.display import display, HTML

def run():
    # Helper to make the input box short without bold text
    def short_input(prompt):
        display(HTML(f"<span>{prompt}</span>"))
        display(HTML("<style>input.widget-text { width: 150px !important; }</style>"))
        return input("> ")

    print("🧞 StudyGenie Started")

    def welcome_screen():
        print("\n" + "="*50)
        print("🧞‍♂️ I am your study genie. Ask your wish before we begin.")
        print("="*50)
        print("\n🎯 Available Wishes:")
        print("1. ❓ Ask Questions")
        print("2. 📄 Generate Summary")
        print("3. 📘 Create Study Plan")
        print("\n📌 First Step: Upload your PDF to awaken the genie...\n")

    welcome_screen()

    while True:
        if not pdf_text:
            ch = short_input("1: Upload PDF / 0: Exit")
            if ch == "1":
                load_pdf()
                build_chunks()
                build_faiss()
            else:
                print("\n🧞 StudyGenie Ended") # Exit message for first prompt
                break
        else:
            print("") # ✅ This adds the single line break you wanted
            ch = short_input("1: Q&A / 2: Summary / 3: Plan / 4: Exit")
            if ch == "1":
                ask_question()
            elif ch == "2":
                summary()
            elif ch == "3":
                study_plan()
            else:
                print("\n🧞 StudyGenie Ended") # ✅ Exit message for main menu
                break

run()